# TD 1 : mise en route et prise en main

## Comment se passe cette journée

Vous travaillez **en binôme**, et vous changez de clavier à chaque étape. Celui qui ne tape pas lit l'énoncé à voix haute et vérifie.

**Personne ne lève la main pour savoir s'il a juste.** Chaque exercice se termine par une cellule de vérification qui vous le dit, et qui explique ce qui ne va pas si ce n'est pas bon. Les indices sont repliés sous chaque exercice : ouvrez-les plutôt que d'attendre.

**Quand vous finissez une étape**, marquez-la sur le tableau au mur. C'est ce qui permet de voir où en est la salle sans faire le tour des rangs.

## Ce que vous saurez faire

1. installer et vérifier votre environnement sans aide
2. charger un vrai fichier de données et dire ce qu'il contient
3. sélectionner, filtrer et agréger avec pandas
4. calculer sur un tableau NumPy sans écrire de boucle
5. produire une figure lisible et dire ce qu'elle montre

## Prérequis

- avoir suivi la journée 1
- savoir ce qu'est une variable et une fonction en Python

**Durée :** 85 minutes

## Le contexte

Vous travaillez aujourd'hui sur **101 766 séjours hospitaliers réels**, en open data, et sur la station de mesure de la pollution vue hier. Pas de jeu jouet : les défauts sont là dès la première cellule, et c'est voulu.

## Le déroulé

| quand | quoi |
|---|---|
| 0:00 | mise en route, environnement, binômes |
| 0:20 | étape 1 : charger et regarder |
| 0:50 | étape 2 : sélectionner, filtrer, agréger |
| 1:20 | étape 3 : NumPy et une première figure |
| 1:45 | pause |

## Avant tout : l'amorce

Exécutez la cellule ci-dessous. Elle règle les chemins et importe ce qu'il faut. **Si elle échoue, levez la main maintenant**, c'est le seul moment de la journée où c'est la bonne chose à faire.

In [ ]:
# Amorce : a executer en premier, une seule fois par notebook.
import sys, pathlib
racine = pathlib.Path.cwd()
while not (racine / "aeolis").is_dir() and racine != racine.parent:
    racine = racine.parent
sys.path.insert(0, str(racine))
import os
os.chdir(racine)

import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aeolis import plots, evaluate as ev

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

# Les donnees sont dans cours_fr/data pendant le cours, et dans donnees/ dans le
# dossier qui vous est remis. On prend celui qui existe, pour que le meme notebook
# fonctionne dans les deux cas.
DONNEES = next((c for c in (racine / "cours_fr" / "data", racine / "donnees",
                            racine / "data") if c.is_dir()), racine)

print("racine du cours :", racine.name)
print("donnees :", DONNEES.name, "->",
      sorted(p.name for p in DONNEES.glob("*.parquet")) or "aucun parquet")
print("pret")

---

## Étape 1 : Charger, et ne pas croire ce qu'on voit

**30 minutes.** Un fichier n'est pas des données. Avant toute statistique, savoir ce qui est réellement arrivé en mémoire.

## Mémo de code

Chaque exercice ci-dessous se résout en adaptant un de ces patrons. Copiez, changez les noms, exécutez. Ce n'est pas tricher : c'est ainsi que l'on écrit du code de data.

```python
# charger un parquet
table = pd.read_parquet(DONNEES / 'readmission.parquet')

# la forme : combien de lignes, combien de colonnes
table.shape

# le type réel de chaque colonne
table.dtypes

# les premieres lignes
table.head(3)

# compter les valeurs d'une colonne
table['readmitted'].value_counts()

# compter une valeur particuliere par colonne
(table == '?').sum()

# la part de valeurs manquantes
table.isna().mean()
```

### 1.1 Charger le fichier et dire ce qu'il contient

In [ ]:
# A FAIRE : charger le fichier et afficher sa forme.
sejours = ...

print('lignes et colonnes :', sejours.shape)

**Indice 1.** Le premier patron du mémo. Le chemin est `DONNEES / 'readmission.parquet'`, ou DONNEES vient de l'amorce.

**Indice 2.** `sejours = pd.read_parquet(DONNEES / 'readmission.parquet')`

In [ ]:
assert sejours.shape == (101766, 50), (
    f"attendu (101766, 50), obtenu {sejours.shape}. "
    'Si le nombre de lignes est bon mais pas celui des colonnes, vous avez '
    'peut-etre selectionne des colonnes sans le vouloir.')
print('correct : 101 766 sejours, 50 colonnes')

### 1.2 Combien de valeurs manquantes ?

Question simple en apparence. Comptez-les de deux façons différentes et comparez les deux nombres.

In [ ]:
# A FAIRE (1) : combien de valeurs manquantes au sens de pandas ?
manquants_pandas = ...

# A FAIRE (2) : combien de cellules valent exactement '?' ?
interrogations = ...

print(f'manquantes selon pandas : {manquants_pandas:,}')
print(f"cellules valant '?'      : {interrogations:,}")

**Indice 1.** Pour (1), `isna()` puis `sum()`, deux fois : une par colonne, puis le total. Pour (2), le patron `(table == '?')` du mémo, même logique.

**Indice 2.** `sejours.isna().sum().sum()` et `(sejours == '?').sum().sum()`

In [ ]:
assert manquants_pandas == 181168, (
    f'attendu 181 168, obtenu {manquants_pandas}. '
    'Avez-vous bien somme deux fois, une fois par colonne puis le total ?')
assert interrogations == 192849, (
    f'attendu 192 849, obtenu {interrogations}.')
print('correct : 181 168 manquantes vues par pandas, et 192 849 de plus')
print("qu'il ne voit pas, parce qu'elles sont ecrites avec un caractere.")
print()
print('Regardez OU se trouve chaque sorte :')
nan_par_colonne = sejours.isna().sum()
q_par_colonne = (sejours == '?').sum()
print('  NaN :', list(nan_par_colonne[nan_par_colonne > 0].index))
print("  '?' :", list(q_par_colonne[q_par_colonne > 0].index))

---

## Étape 2 : Sélectionner, filtrer, agréger

**30 minutes.** Les trois opérations qui couvrent 90 % du travail réel sur un tableau.

### 2.1 Quelles colonnes sont inutilisables ?

Vous savez qu'il y a 192 849 `?`. Ils ne sont pas répartis uniformément.

In [ ]:
# A FAIRE : la part de '?' par colonne, pour les colonnes concernees,
#           triee de la pire a la moins pire.
part_interrogations = ...

print(part_interrogations.head(6))

**Indice 1.** `(sejours == '?').mean()` donne une part par colonne. Il reste à garder celles qui sont non nulles et à trier.

**Indice 2.** `part = (sejours == '?').mean()` puis `part[part > 0].sort_values(ascending=False)`

In [ ]:
assert len(part_interrogations) == 7, (
    f'7 colonnes contiennent des ?, vous en trouvez {len(part_interrogations)}')
pire = part_interrogations.index[0]
assert pire == 'weight', f"la pire colonne est 'weight', vous trouvez '{pire}'"
assert abs(part_interrogations.iloc[0] - 0.9686) < 0.001
print(f'correct. La pire est {pire}, vide a '
      f'{part_interrogations.iloc[0]:.1%}.')
print('Une colonne vide a 97 % ne se repare pas. Elle se supprime.')

### 2.2 Le taux de réadmission

La colonne `readmitted` vaut `NO`, `>30` ou `<30`. Seul `<30` nous intéresse : une réadmission sous 30 jours.

In [ ]:
# A FAIRE (1) : compter les trois valeurs de 'readmitted'.
repartition = ...

# A FAIRE (2) : la part de sejours suivis d'une readmission sous 30 jours.
taux = ...

print(repartition)
print(f'\ntaux de readmission sous 30 jours : {taux:.4f}')

**Indice 1.** Pour (1), le patron `value_counts()`. Pour (2), comparer la colonne à `'<30'` donne des booléens, et la moyenne de booléens est une proportion.

**Indice 2.** `sejours['readmitted'].value_counts()` et `(sejours['readmitted'] == '<30').mean()`

In [ ]:
assert abs(taux - 0.1116) < 0.001, (
    f'attendu environ 0,1116, obtenu {taux:.4f}. '
    'Il y a trois modalites : NO, >30 et <30. Une seule nous interesse.')
print(f'correct : {taux:.2%} des sejours.')
print(f'Donc répondre toujours non donne {1 - taux:.2%} d accuracy.')
print('Retenez ce nombre : le TD 3 en fait son point de depart.')

---

## Étape 3 : NumPy et une première figure

**25 minutes.** Calculer sur un tableau sans boucle, et produire une figure dont on peut dire ce qu'elle montre.

### 3.1 Sans boucle

`time_in_hospital` est la durée du séjour en jours. Elle est arrivée en texte, comme tout le reste de ce fichier.

In [ ]:
# A FAIRE : convertir en nombres, puis moyenne, mediane et ecart-type.
duree = ...

print(f'moyenne  {duree.mean():.2f} jours')
print(f'mediane  {np.median(duree):.2f} jours')
print(f'ecart-type {duree.std():.2f} jours')

**Indice 1.** `pd.to_numeric` convertit une colonne de texte en nombres. `.values` donne le tableau NumPy derrière.

**Indice 2.** `duree = pd.to_numeric(sejours['time_in_hospital']).values`

In [ ]:
assert duree.dtype.kind in 'if', (
    f'duree doit etre numerique, son type est {duree.dtype}. '
    'Avez-vous oublie pd.to_numeric ?')
assert abs(duree.mean() - 4.396) < 0.01
assert abs(np.median(duree) - 4.0) < 0.01
print('correct. Moyenne 4,40 jours, mediane 4,00 jours.')
print("L'ecart entre les deux dit que la distribution est asymetrique :")
print('quelques sejours tres longs tirent la moyenne vers le haut.')

### 3.2 La figure, et ce qu'elle montre

In [ ]:
# A FAIRE : un histogramme des durees de sejour, 14 classes,
#           avec les deux axes nommes et un titre.
plt.figure(figsize=(9, 3.2))
...

axe = plt.gca()        # on garde la reference pour la verification
plt.show()

**Indice 1.** `plt.hist(duree, bins=14)`, puis `plt.xlabel`, `plt.ylabel`, `plt.title`. Une figure sans axes nommés n'est pas une figure, c'est un dessin.

**Indice 2.** `plt.hist(duree, bins=14, edgecolor='white')` puis les trois appels d'étiquetage.

In [ ]:
assert axe.get_xlabel(), "l'axe des x n'a pas de nom"
assert axe.get_ylabel(), "l'axe des y n'a pas de nom"
assert axe.get_title(), 'la figure n a pas de titre'
assert len(axe.patches) == 14, (
    f'attendu 14 classes, la figure en a {len(axe.patches)}')
print('correct : 14 classes, axes nommes, titre present.')
print('Regardez la forme : un pic vers 3 jours et une longue queue a droite.')
print("C'est exactement ce que l'ecart moyenne/mediane annoncait.")

## Ce que cette étape a établi

- Un fichier n'est pas des données : `dtypes` et un regard avant toute statistique.
- 192 849 cellules valent `?` et pandas n'en voit aucune. Un marqueur de valeur manquante est une convention locale.
- Une colonne vide à 96,9 % se supprime. Une colonne vide à 49,1 % se discute.
- 11,16 % de réadmissions, donc 88,84 % d'accuracy sans rien apprendre.
- Moyenne 4,40 contre médiane 4,00 : la distribution est asymétrique, et la figure le montre.

## Les erreurs classiques

- **Lancer `describe()` avant `dtypes`.** Les colonnes texte sont ignorées en silence, et vous ne le saurez jamais.
- **Remplir une colonne vide à 97 % par la moyenne.** C'est inventer 98 569 valeurs à partir de 3 197. La colonne se supprime.
- **Produire une figure sans nommer les axes.** Personne ne peut la lire, vous compris, dans trois semaines.

## Pour aller plus loin, si vous avez fini en avance

- Regarder `discharge_disposition_id` avec `value_counts()`. Certaines valeurs vont poser un problème très sérieux au TD 3, et vous pouvez le trouver maintenant.
- Comparer la durée de séjour des patients réadmis et des autres. L'écart est-il grand ?